In [ ]:
import anndata
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scanpy as sc
import seaborn as sns
import shutil
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.mixture import GaussianMixture

from mcDETECT.downstream import *

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [ ]:
# (1) Use Isocortex or HPF, (2) Use 25×25 or 50×50 grids
ROI = "Isocortex"
# grid_len = "25_by_25"
grid_len = "50_by_50"
grid_len_num = int(grid_len.split("_")[0])

# File paths
data_path_wt = f"../data/MERSCOPE_WT_1/processed_data/"
data_path_ad = f"../data/MERSCOPE_AD_1/processed_data/"
comparison_path = f"../output/MERSCOPE_WT_AD_comparison/"
output_path = f"../output/MERSCOPE_WT_AD_comparison/" + f"neuropil_subdomains_{ROI}_{grid_len_num}/"
# shutil.rmtree(output_path, ignore_errors=True)
os.makedirs(output_path, exist_ok=True)

# Plot settings
plot_settings = {"Isocortex": {"25_by_25": {"figure_size": (12.5, 6.5), "spot_size": 25}, "50_by_50": {"figure_size": (12.5, 5.5), "spot_size": 80}},
                 "HPF": {"25_by_25": {"figure_size": (10, 7), "spot_size": 18}, "50_by_50": {"figure_size": (10, 7), "spot_size": 36}}}

# Color map
color_dct = ["#F56867","#FEB915","#C798EE","#59BE86","#7495D3","#997273"]

# Pre-syn score settings
subtype_names = ["pre-syn", "post-syn", "dendrites", "mixed"]
condition_col = "batch"
wt_label = "MERSCOPE_WT_1"
ad_label = "MERSCOPE_AD_1"

# Benchmark clustering settings
k_range = range(2, 10)
random_state = 42
batch_size = 1000
n_seeds = 5
stability_seeds = [0, 42, 123, 456, 789][:n_seeds]

# Heatmap settings
granule_col_order = ["0", "11", "12", "13", "1", "2", "3", "4", "8", "5", "6", "7", "9", "10", "14"]
subtype_to_category = {"0": "pre-syn", "11": "pre-syn", "12": "pre-syn", "13": "pre-syn", "1": "post-syn", "2": "post-syn", "3": "post-syn", "4": "dendrites", "8": "dendrites", "5": "mixed", "6": "mixed", "7": "mixed", "9": "mixed", "10": "mixed", "14": "mixed"}
ordered_cats = ["pre-syn", "post-syn", "dendrites", "mixed"]
palette = sns.color_palette("Set2", n_colors=len(ordered_cats))
cat_to_color = dict(zip(ordered_cats, palette))

In [ ]:
# Recover 50×50 grid annotation
def recover_spots(spots, roi, batch = "MERSCOPE_WT_1"):
    
    # Read reference spots
    spots_ref = sc.read_h5ad(comparison_path + f"neuropil_subdomains_spots_{roi}.h5ad")
    
    # Select spots
    if roi == "Isocortex":
        if batch == "MERSCOPE_WT_1":
            select_mask = spots.obs["region_labels"].isin([1, 3, 15, 12, 13, 23])
        elif batch == "MERSCOPE_AD_1":
            select_mask = spots.obs["region_labels"].isin([2, 5, 11, 17, 26])
        spots_select = spots[select_mask].copy()
    elif roi == "HPF":
        select_mask = spots.obs["brain_area"].isin(["HPF-CA", "HPF-DG", "HPF-SR"])
        spots_select = spots[select_mask].copy()
    
    # Recover annotation
    labels_df = pd.DataFrame({"global_x": spots_ref.obs["global_x"].to_numpy(), "global_y": spots_ref.obs["global_y"].to_numpy(), "layer_labels": spots_ref.obs["layer_labels"].to_numpy()}).reset_index(drop=True)
    ref_xy = labels_df[["global_x", "global_y"]].to_numpy()
    tree = cKDTree(ref_xy)
    if batch == "MERSCOPE_WT_1":
        spots_xy = spots_select.obs[["global_y_new", "global_x_new"]].to_numpy()
    elif batch == "MERSCOPE_AD_1":
        spots_select.obs["global_x_new"] += 12000
        spots_select.obs["global_y_new"] += 7200
        spots_xy = spots_select.obs[["global_x_new", "global_y_new"]].to_numpy()
    _, nn_idx = tree.query(spots_xy, k=1)
    spots_select = spots_select.copy()
    spots_select.obs["layer_labels"] = labels_df.loc[nn_idx, "layer_labels"].to_numpy()
    
    spots.obs["layer_labels"] = None
    spots.obs.loc[select_mask, "layer_labels"] = spots_select.obs["layer_labels"].to_numpy()
    return spots

In [ ]:
# ==================== Read data ==================== #

# Spots
if grid_len == "25_by_25":
    
    spots = sc.read_h5ad(comparison_path + f"neuropil_subdomains_spots_{ROI}.h5ad")
    spots = spots[(spots.obs["layer_labels"].notna()) & (spots.obs["layer_labels"] != "Others")].copy()
    
elif grid_len == "50_by_50":
    
    spots_wt = sc.read_h5ad(data_path_wt + "spots.h5ad")
    spots_ad = sc.read_h5ad(data_path_ad + "spots.h5ad")
    
    spots_wt = recover_spots(spots_wt, ROI, batch = "MERSCOPE_WT_1")
    spots_ad = recover_spots(spots_ad, ROI, batch = "MERSCOPE_AD_1")
    
    spots_wt.obs["global_x"] = spots_wt.obs["global_y_new"].copy()
    spots_wt.obs["global_y"] = spots_wt.obs["global_x_new"].copy()

    spots_ad.obs["global_x"] = spots_ad.obs["global_x_new"].copy()
    spots_ad.obs["global_y"] = spots_ad.obs["global_y_new"].copy()
    spots_ad.obs["global_x"] += 12000
    spots_ad.obs["global_y"] += 7200

    spots_dict = {"MERSCOPE_WT_1": spots_wt, "MERSCOPE_AD_1": spots_ad}
    spots = anndata.concat(spots_dict, axis = 0, merge = "same", label = "batch")
    spots = spots[spots.obs["brain_area"] != "Unknown"].copy()
    spots.obs.loc[spots.obs["layer_labels"] == "Others", "layer_labels"] = None
    
    sc.set_figure_params(scanpy = True, figsize = (10, 7))
    ax = sc.pl.scatter(spots, x="global_x", y="global_y", color="layer_labels", palette=color_dct, size=45, title=" ", show = False)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.savefig(output_path + f"ROI.jpeg", dpi = 500, bbox_inches = "tight")
    plt.close()
    
    spots = spots[spots.obs["layer_labels"].notna()].copy()

# Cells
adata = sc.read_h5ad(comparison_path + "neuropil_subdomains_adata.h5ad")

# Cell count matrix
count_matrix_wt = pd.read_parquet(data_path_wt + "count.parquet")
count_matrix_ad = pd.read_parquet(data_path_ad + "count.parquet")

# Combined count matrix used for mapping by cell_id
count_matrix = pd.concat([count_matrix_wt, count_matrix_ad], axis=0, ignore_index=True)

# Granules
granule_adata = sc.read_h5ad(comparison_path + "neuropil_subdomains_granule_adata.h5ad")

# # Plot check
# sc.pl.scatter(spots, x="global_x", y="global_y", color="layer_labels")
# plt.show()

# sc.pl.scatter(adata, x="global_x", y="global_y", color="brain_area")
# plt.show()

sc.set_figure_params(scanpy = True, figsize = plot_settings[ROI][grid_len]["figure_size"])
ax = sc.pl.scatter(spots, x="global_x", y="global_y", color="layer_labels", palette=color_dct, size=plot_settings[ROI][grid_len]["spot_size"], title=" ", show = False)
ax.grid(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")
for spine in ax.spines.values():
    spine.set_visible(False)
plt.savefig(output_path + f"ROI_layers.jpeg", dpi = 500, bbox_inches = "tight")
plt.close()

In [ ]:
# Align count matrix gene order to granule_adata.var_names
if "cell_id" not in count_matrix.columns:
    raise ValueError("count_matrix must contain a 'cell_id' column")

_gene_order = list(granule_adata.var_names)
count_matrix = count_matrix.copy()
count_matrix["cell_id"] = count_matrix["cell_id"].astype(str)
count_matrix = count_matrix.set_index("cell_id").reindex(columns=_gene_order, fill_value=0).reset_index()
print(count_matrix.columns[1:].tolist() == granule_adata.var_names.tolist())

In [ ]:
# Previous neuron-granule counts
adata_neuron = adata[(adata.obs["brain_area"].str.startswith(ROI)) & (adata.obs["cell_type"].isin(["GABAergic", "Glutamatergic"]))].copy()
_, _, granule_counts_neuron = neuron_embedding_spatial_weight(adata_neuron, granule_adata, radius = 10, sigma = 10, loc_key = ["global_x", "global_y"], gnl_subtype_key = "granule_subtype_kmeans", padding_value = "Others")
print(f"Mean neuron-granule counts: {np.nanmean(granule_counts_neuron):.3f}, Median neuron-granule counts: {np.nanmedian(granule_counts_neuron):.3f}")

In [ ]:
# # ==================== Pre-syn scores ==================== #

# # 0. Calculate embeddings
# embeddings, embeddings_features, aux_features, *_ = spot_embedding(
#         spots=spots,
#         granule_adata=granule_adata,
#         adata=adata_neuron,
#         count_matrix=count_matrix,
#         spot_loc_key=("global_x", "global_y"),
#         spot_width=grid_len_num,
#         spot_height=grid_len_num,
#         granule_loc_key=("global_x", "global_y"),
#         granule_subtype_key="granule_subtype_manual_simple",
#         subtype_names=subtype_names,
#         granule_count_layer="counts",
#         cell_loc_key=("global_x", "global_y"),
#         cell_id_key="cell_id",
#         count_matrix_cell_id_key="cell_id",
#         include_soma_features=True,
#         smoothing=False,
#         smoothing_radius=np.sqrt(2) * grid_len_num + 1,
#         smoothing_mode="gaussian"
#     )

# for aux_key, aux_val in aux_features.items():
#     spots.obs[aux_key] = aux_val

# # Remove spots with no granules
# mask = (spots.obs["granule_count"] > 0)
# spots = spots[mask].copy()
# embeddings = embeddings[mask].copy()

# # ---------- 1. Spot-level subtype proportions ---------- #
# row_sums = embeddings.sum(axis=1, keepdims=True)
# X_prop = np.divide(embeddings, row_sums, out=np.zeros_like(embeddings, dtype=float), where=row_sums > 0)
# for j, s in enumerate(subtype_names):
#     spots.obs[f"prop_{s}"] = X_prop[:, j]

# # ---------- 2. Continuous enrichment scores (enrichment relative to WT baseline) ---------- #
# wt_mask = spots.obs[condition_col].to_numpy() == wt_label
# ad_mask = spots.obs[condition_col].to_numpy() == ad_label

# if wt_mask.sum() == 0:
#     raise ValueError(f"No spots found with {condition_col} == '{wt_label}'")
# if ad_mask.sum() == 0:
#     raise ValueError(f"No spots found with {condition_col} == '{ad_label}'")

# wt_mean_prop = X_prop[wt_mask].mean(axis=0)
# wt_baseline_df = pd.DataFrame({"subtype": subtype_names, "wt_mean_prop": wt_mean_prop})

# eps = 1e-6
# enrichment_vs_wt_scores = np.log(X_prop + eps) - np.log(wt_mean_prop[None, :] + eps)

# for j, s in enumerate(subtype_names):
#     spots.obs[f"enrich_vs_wt_{s}"] = enrichment_vs_wt_scores[:, j]

# # ---------- optional: a one-vs-rest score ---------- #
# pre_idx = subtype_names.index("pre-syn")
# post_idx = subtype_names.index("post-syn")
# den_idx = subtype_names.index("dendrites")
# mixed_idx = subtype_names.index("mixed")

# enrichment_vs_rest_scores = []
# for target_idx, target_label in zip([pre_idx, post_idx, den_idx, mixed_idx], ["pre", "post", "den", "mix"]):
#     other_idx = [i for i in range(len(subtype_names)) if i != target_idx]
#     score = np.log(X_prop[:, target_idx] + eps) - np.log(X_prop[:, other_idx].sum(axis=1) + eps)
#     spots.obs[f"score_{target_label}_vs_rest"] = score
#     enrichment_vs_rest_scores.append(score)
# enrichment_vs_rest_scores = np.vstack(enrichment_vs_rest_scores).T

# # ---------- optional: a pre-vs-post score ---------- #
# pre_vs_post_score = np.log(X_prop[:, pre_idx] + eps) - np.log(X_prop[:, post_idx] + eps)
# spots.obs["score_pre_vs_post"] = pre_vs_post_score

# # ---------- optional: subtype with strongest enrichment ---------- #
# dominant_enrichment_vs_wt_idx = enrichment_vs_wt_scores.argmax(axis=1)
# spots.obs["dominant_enrichment_vs_wt"] = pd.Categorical(
#     [subtype_names[i] for i in dominant_enrichment_vs_wt_idx],
#     categories=subtype_names
# )

# dominant_enrichment_vs_rest_idx = enrichment_vs_rest_scores.argmax(axis=1)
# spots.obs["dominant_enrichment_vs_rest"] = pd.Categorical(
#     [subtype_names[i] for i in dominant_enrichment_vs_rest_idx],
#     categories=subtype_names
# )

# # ---------- optional: top xxx% pre-syn-enriched spots ---------- #
# q = 0.8

# threshold = np.quantile(spots.obs.loc[wt_mask, "enrich_vs_wt_pre-syn"], q)
# spots.obs[f"top_{round(1-q, 1)}_presyn_enriched_vs_wt"] = (spots.obs["enrich_vs_wt_pre-syn"] >= threshold).astype(str)

# threshold = np.quantile(spots.obs.loc[wt_mask, "score_pre_vs_rest"], q)
# spots.obs[f"top_{round(1-q, 1)}_presyn_enriched_vs_rest"] = (spots.obs["score_pre_vs_rest"] >= threshold).astype(str)

# # ---------- 3. Visualization ---------- #
# for cluster_col in ["prop_pre-syn"]:
#     sc.set_figure_params(scanpy = True, figsize = plot_settings[ROI][grid_len]["figure_size"])
#     ax = sc.pl.scatter(spots, x="global_x", y="global_y", color=cluster_col, color_map="Reds", size=plot_settings[ROI][grid_len]["spot_size"], title=" ", show = False)
#     ax.grid(False)
#     ax.set_xticks([])
#     ax.set_yticks([])
#     ax.set_xlabel("")
#     ax.set_ylabel("")
#     ax.set_title("")
#     for spine in ax.spines.values():
#         spine.set_visible(False)
#     plt.savefig(output_path + f"{cluster_col}.jpeg", dpi = 500, bbox_inches = "tight")
#     plt.close()

# for cluster_col in ["enrich_vs_wt_pre-syn", "score_pre_vs_rest", "score_pre_vs_post"]:
    
#     vals = spots.obs[cluster_col].to_numpy()
#     lim = np.nanmax(np.abs(vals))
#     clipped_col = f"{cluster_col}_clipped"
#     spots.obs[clipped_col] = np.clip(vals, -lim, lim)
    
#     sc.set_figure_params(scanpy = True, figsize = plot_settings[ROI][grid_len]["figure_size"])
#     ax = sc.pl.scatter(spots, x="global_x", y="global_y", color=clipped_col, color_map="bwr", size=plot_settings[ROI][grid_len]["spot_size"], title=" ", show = False)
#     ax.grid(False)
#     ax.set_xticks([])
#     ax.set_yticks([])
#     ax.set_xlabel("")
#     ax.set_ylabel("")
#     ax.set_title("")
#     for spine in ax.spines.values():
#         spine.set_visible(False)
#     plt.savefig(output_path + f"{cluster_col}.jpeg", dpi = 500, bbox_inches = "tight")
#     plt.close()
    
#     del spots.obs[clipped_col]

# for cluster_col in ["dominant_enrichment_vs_wt", "dominant_enrichment_vs_rest", "top_0.2_presyn_enriched_vs_wt", "top_0.2_presyn_enriched_vs_rest"]:
#     sc.set_figure_params(scanpy = True, figsize = plot_settings[ROI][grid_len]["figure_size"])
#     ax = sc.pl.scatter(spots, x="global_x", y="global_y", color=cluster_col, size=plot_settings[ROI][grid_len]["spot_size"], title=" ", show = False)
#     ax.grid(False)
#     ax.set_xticks([])
#     ax.set_yticks([])
#     ax.set_xlabel("")
#     ax.set_ylabel("")
#     ax.set_title("")
#     for spine in ax.spines.values():
#         spine.set_visible(False)
#     plt.savefig(output_path + f"{cluster_col}.jpeg", dpi = 500, bbox_inches = "tight")
#     plt.close()

In [ ]:
# ==================== Main operation ==================== #

spots0 = spots.copy()

# -------------------- Embedding -------------------- #
for embedding_method in ["hard", "soft"]:
    
    spots = spots0.copy()
    
    if embedding_method == "hard":
        
        embeddings, embeddings_features, aux_features, spot_granule_expression, spot_cell_expression = spot_embedding(
            spots=spots,
            granule_adata=granule_adata,
            adata=adata_neuron,
            count_matrix=count_matrix,
            spot_loc_key=("global_x", "global_y"),
            spot_width=grid_len_num,
            spot_height=grid_len_num,
            granule_loc_key=("global_x", "global_y"),
            granule_subtype_key="granule_subtype_kmeans",
            subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
            granule_count_layer="counts",
            cell_loc_key=("global_x", "global_y"),
            cell_id_key="cell_id",
            count_matrix_cell_id_key="cell_id",
            include_soma_features=True,
            smoothing=True,
            smoothing_radius=np.sqrt(2) * grid_len_num + 1,
            smoothing_mode="gaussian"
        )
    
    elif embedding_method == "soft":
        
        # skip soft embedding for now
        continue
        
        embeddings, embeddings_features, aux_features, spot_granule_expression, spot_cell_expression = spot_embedding_soft(
            spots=spots,
            granule_adata=granule_adata,
            adata=adata_neuron,
            count_matrix=count_matrix,
            spot_loc_key=("global_x", "global_y"),
            spot_width=grid_len_num,
            spot_height=grid_len_num,
            granule_loc_key=("global_x", "global_y"),
            granule_subtype_key="granule_subtype_kmeans",
            subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
            granule_count_layer="counts",
            cell_loc_key=("global_x", "global_y"),
            cell_id_key="cell_id",
            count_matrix_cell_id_key="cell_id",
            include_soma_features=True,
            kernel="exponential",
            sigma=grid_len_num / 2,
            support_radius=grid_len_num,
            normalize_subtype_embedding=False,
            normalize_gene_counts=False,
        )
    
    for aux_key, aux_val in aux_features.items():
        spots.obs[aux_key] = aux_val
    
    # Rough comparison with neuron-granule counts
    granule_counts = spots.obs["granule_count"].to_numpy()
    print(f"Mean spot-granule counts: {np.nanmean(granule_counts):.3f}, Median spot-granule counts: {np.nanmedian(granule_counts):.3f}")
    soma_counts = spots.obs["soma_count"].to_numpy()
    print(f"Mean spot-neuron counts: {np.nanmean(soma_counts):.3f}, Median spot-neuron counts: {np.nanmedian(soma_counts):.3f}")
    
    xmax = np.max([np.max(granule_counts), np.max(granule_counts_neuron)])
    
    plt.figure(figsize=(8, 8))
    plt.hist(granule_counts, bins=50, alpha=0.7, edgecolor="black")
    plt.xlim(right=xmax * 1.1)
    plt.grid(False)
    plt.xlabel("Number of Granules")
    plt.ylabel("Count")
    plt.title(" ")
    plt.savefig(output_path + f"{embedding_method}_granule_counts.jpeg", dpi=500, bbox_inches="tight")
    plt.close()
    
    plt.figure(figsize=(8, 8))
    plt.hist(granule_counts_neuron, bins=50, color="#9b59b6", alpha=0.7, edgecolor="black")
    plt.xlim(right=xmax)
    plt.grid(False)
    plt.xlabel("Number of Granules")
    plt.ylabel("Count")
    plt.title(" ")
    plt.savefig(output_path + "granule_counts_per_neuron.jpeg", dpi=500, bbox_inches="tight")
    plt.close()
    
    # Remove spots with no granules
    mask = (spots.obs["granule_count"] > 0)
    spots = spots[mask].copy()
    embeddings = embeddings[mask].copy()
    spot_granule_expression = spot_granule_expression[mask].copy()
    spot_cell_expression = spot_cell_expression[mask].copy()

    # -------------------- Clustering -------------------- #
    for mode in ["unnormalized", "normalized"]:
        
        if mode == "normalized":
            row_sums = embeddings.sum(axis=1, keepdims=True)
            X = np.divide(embeddings, row_sums, out=np.zeros_like(embeddings, dtype=float), where=row_sums > 0)
        elif mode == "unnormalized":
            # skip unnormalized mode for now
            continue
            X = embeddings
        
        # Justify number of clusters using (1) Minibatch K-Means and (2) K = 5
        print(f"Benchmarking number of clusters using {embedding_method} embedding and {mode} mode... ----------")
        
        results = []
        for k in k_range:
            km = MiniBatchKMeans(n_clusters=k, random_state=random_state, batch_size=batch_size)
            labels = km.fit_predict(X)
            inertia = km.inertia_
            label_list = [labels]
            for seed in stability_seeds[1:]:
                km_r = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=batch_size)
                label_list.append(km_r.fit_predict(X))
            ari_values = []
            for i in range(len(label_list)):
                for j in range(i + 1, len(label_list)):
                    ari_values.append(adjusted_rand_score(label_list[i], label_list[j]))
            ari_stability_mean = np.mean(ari_values) if ari_values else np.nan
            results.append({"n_clusters": k, "inertia": inertia, "ari_stability_mean": ari_stability_mean})
        metrics_df = pd.DataFrame(results)
        metrics_df.to_csv(output_path + f"{embedding_method}_{mode}_benchmark_clustering_results.csv", index=False)
        
        plt.figure(figsize=(8, 8))
        plt.plot(metrics_df["n_clusters"], metrics_df["inertia"], marker="o", markersize=6)
        plt.xlabel("Number of Clusters")
        plt.ylabel("Inertia")
        plt.title(" ")
        plt.savefig(output_path + f"{embedding_method}_{mode}_benchmark_clustering_results.jpeg", dpi=500, bbox_inches="tight")
        plt.close()
        
        for n_clusters in [4, 5]:
            
            # skip K = 5 for now
            if n_clusters == 5:
                continue
            
            print(f"---------- Embedding method: {embedding_method}, Mode: {mode}, Number of clusters: {n_clusters} ----------")
            
            # LDA clustering
            lda = LatentDirichletAllocation(n_components = n_clusters, random_state = 42)
            lda_labels = lda.fit_transform(X)
            spots.obs["subdomain_lda"] = [f"Subdomain {l + 1}" for l in np.argmax(lda_labels, axis = 1)]
            
            # GMM clustering
            gmm = GaussianMixture(n_components = n_clusters, random_state = 42)
            gmm_labels = gmm.fit_predict(X)
            spots.obs["subdomain_gmm"] = [f"Subdomain {l + 1}" for l in gmm_labels]
            
            # K-Means clustering
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
            kmeans_labels = kmeans.fit_predict(X)
            spots.obs["subdomain_kmeans"] = [f"Subdomain {l + 1}" for l in kmeans_labels]
            
            # Adjust k-means labels
            # relabel_map = {"Subdomain 4": "Subdomain 1", "Subdomain 2": "Subdomain 2", "Subdomain 3": "Subdomain 3", "Subdomain 1": "Subdomain 4"}
            relabel_map = {"Subdomain 3": "Subdomain 1", "Subdomain 2": "Subdomain 2", "Subdomain 1": "Subdomain 3", "Subdomain 4": "Subdomain 4"}
            spots.obs["subdomain_kmeans"] = (spots.obs["subdomain_kmeans"].astype(str).replace(relabel_map))
            desired_order = [f"Subdomain {i + 1}" for i in range(n_clusters)]
            spots.obs["subdomain_kmeans"] = pd.Categorical(spots.obs["subdomain_kmeans"], categories = desired_order, ordered = True)
            
            # Minibatch K-Means clustering
            kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, random_state=42, n_init=20)
            kmeans_labels = kmeans.fit_predict(X)
            spots.obs["subdomain_minibatch"] = [f"Subdomain {l + 1}" for l in kmeans_labels]
            
            # Save cluster labels
            cluster_labels_df = spots.obs[["spot_id", "layer_labels", "subdomain_lda", "subdomain_gmm", "subdomain_kmeans", "subdomain_minibatch"]].copy()
            cluster_labels_df.to_parquet(output_path + f"{n_clusters}_{embedding_method}_{mode}_cluster_labels.parquet")

            # -------------------- Visualization -------------------- #
            for cluster_col in ["subdomain_lda", "subdomain_gmm", "subdomain_kmeans", "subdomain_minibatch"]:
                
                # Delete previous uns
                spots.uns.pop(f"{cluster_col}_colors", None)
                
                # Scatter plot
                sc.set_figure_params(scanpy = True, figsize = plot_settings[ROI][grid_len]["figure_size"])
                ax = sc.pl.scatter(spots, x="global_x", y="global_y", color=cluster_col, size=plot_settings[ROI][grid_len]["spot_size"], title=" ", show = False)
                ax.grid(False)
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_xlabel("")
                ax.set_ylabel("")
                ax.set_title("")
                for spine in ax.spines.values():
                    spine.set_visible(False)
                plt.savefig(output_path + f"{n_clusters}_{embedding_method}_{mode}_{cluster_col.split('_')[1]}.jpeg", dpi = 500, bbox_inches = "tight")
                plt.close()
                
                for cluster in spots.obs[cluster_col].unique():
                    sc.set_figure_params(scanpy = True, figsize = plot_settings[ROI][grid_len]["figure_size"])
                    ax = sc.pl.scatter(spots[spots.obs[cluster_col] == cluster], x="global_x", y="global_y", color=cluster_col, size=plot_settings[ROI][grid_len]["spot_size"], title=" ", show = False)
                    ax.grid(False)
                    ax.set_xticks([])
                    ax.set_yticks([])
                    ax.set_xlabel("")
                    ax.set_ylabel("")
                    ax.set_title("")
                    for spine in ax.spines.values():
                        spine.set_visible(False)
                    plt.savefig(output_path + f"{n_clusters}_{embedding_method}_{mode}_{cluster_col.split('_')[1]}_{cluster}.jpeg", dpi = 500, bbox_inches = "tight")
                    plt.close()
                
                # Embedding heatmap (mean value)
                all_subtype_labels = [str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
                desired_order = [f"Subdomain {i + 1}" for i in range(n_clusters)]
                embedding_df = pd.DataFrame(X, columns=all_subtype_labels)
                embedding_df[cluster_col] = spots.obs[cluster_col].values
                cluster_means = embedding_df.groupby(cluster_col).mean()
                cluster_means = cluster_means.loc[desired_order]

                col_order = [s for s in granule_col_order if s in cluster_means.columns]
                cluster_means = cluster_means[col_order]
                category_series = pd.Series([subtype_to_category.get(s, "mixed") for s in col_order], index=col_order, name="Category")
                col_colors = category_series.map(cat_to_color)
                
                g = sns.clustermap(cluster_means, cmap="Reds", annot=False, col_colors=col_colors, row_cluster=False, col_cluster=False, linewidths=0.5, linecolor="lightgray", xticklabels=True, yticklabels=True, figsize=(10, 2.5), cbar_pos=(0.02, 0.2, 0.02, 0.5))
                g.ax_heatmap.set_position([0.1, 0.2, 0.75, 0.7])
                g.cax.set_position([0.88, 0.2, 0.02, 0.5])
                g.ax_col_colors.set_position([0.1, 0.92, 0.75, 0.04])
                g.ax_heatmap.tick_params(axis="both", which="both", length=0)
                g.ax_heatmap.grid(False)
                g.ax_heatmap.set_xlabel("Granule Subtype")
                g.ax_heatmap.set_ylabel("Neuropil Subdomain", labelpad=10)
                g.ax_heatmap.yaxis.set_label_position("left")
                g.ax_heatmap.yaxis.tick_left()
                g.ax_heatmap.set_title(" ")
                g.ax_row_dendrogram.set_visible(False)
                g.ax_col_dendrogram.set_visible(False)
                for spine in g.ax_col_colors.spines.values():
                    spine.set_visible(False)
                g.cax.grid(False)
                g.ax_col_colors.set_xticks([])
                g.ax_col_colors.set_yticks([])
                g.ax_col_colors.grid(False)
                g.ax_heatmap.spines["top"].set_visible(False)
                plt.savefig(output_path + f"{n_clusters}_{embedding_method}_{mode}_heatmap_{cluster_col.split('_')[1]}.jpeg", dpi=500, bbox_inches="tight")
                plt.close()
                
                # Embedding heatmap (log2FC)
                if mode == "normalized":
                    global_means = embedding_df[col_order].mean(axis=0)
                    eps = 1e-8
                    log2fc_mat = np.log2((cluster_means + eps).divide(global_means + eps, axis=1))

                    vmax = np.nanmax(np.abs(log2fc_mat.values))
                    vmin = -vmax

                    g = sns.clustermap(log2fc_mat, cmap="bwr", center=0, vmin=vmin, vmax=vmax, annot=False, col_colors=col_colors, row_cluster=False, col_cluster=False, linewidths=0.5, linecolor="lightgray", xticklabels=True, yticklabels=True, figsize=(10, 3), cbar_pos=(0.02, 0.2, 0.02, 0.5))
                    g.ax_heatmap.set_position([0.1, 0.2, 0.75, 0.7])
                    g.cax.set_position([0.88, 0.2, 0.02, 0.5])
                    g.ax_col_colors.set_position([0.1, 0.92, 0.75, 0.04])
                    g.ax_heatmap.tick_params(axis="both", which="both", length=0)
                    g.ax_heatmap.grid(False)
                    g.ax_heatmap.set_xlabel("Granule Subtype")
                    g.ax_heatmap.set_ylabel("Neuropil Subdomain", labelpad=10)
                    g.ax_heatmap.yaxis.set_label_position("left")
                    g.ax_heatmap.yaxis.tick_left()
                    g.ax_heatmap.set_title(" ")
                    g.ax_row_dendrogram.set_visible(False)
                    g.ax_col_dendrogram.set_visible(False)
                    for spine in g.ax_col_colors.spines.values():
                        spine.set_visible(False)
                    g.cax.grid(False)
                    g.ax_col_colors.set_xticks([])
                    g.ax_col_colors.set_yticks([])
                    g.ax_col_colors.grid(False)
                    g.ax_heatmap.spines["top"].set_visible(False)
                    plt.savefig(output_path + f"{n_clusters}_{embedding_method}_{mode}_heatmap_log2fc_{cluster_col.split('_')[1]}.jpeg", dpi=500, bbox_inches="tight")
                    plt.close()
                
                # -------------------- DE analysis -------------------- #
                if cluster_col == "subdomain_kmeans":
                    
                    target_cluster, reference_cluster = "Subdomain 1", "Subdomain 2"
                    mask = spots.obs[cluster_col].isin([target_cluster, reference_cluster])

                    spot_granule_expression_select = spot_granule_expression[mask].copy()
                    spot_cell_expression_select = spot_cell_expression[mask].copy()
                    obs_select = spots.obs.loc[mask].copy()

                    for spot_expression, label in zip([spot_granule_expression_select, spot_cell_expression_select], ["granule", "cell"]):
                        
                        spot_expression_adata = anndata.AnnData(X = spot_expression.copy(), obs = obs_select.copy(), var = granule_adata.var.copy())
                        sc.pp.normalize_total(spot_expression_adata, target_sum = 1e4)
                        sc.pp.log1p(spot_expression_adata)
                        
                        sc.tl.rank_genes_groups(spot_expression_adata, groupby = cluster_col, groups = [target_cluster], reference = reference_cluster, method = "wilcoxon")
                        df = sc.get.rank_genes_groups_df(spot_expression_adata, group=target_cluster)
                        df = df.sort_values("logfoldchanges", ascending=False)
                        df.to_csv(output_path + f"{label}_DE_genes_{target_cluster}_vs_{reference_cluster}.csv", index = False)
                
                # # -------------------- Enrichment analysis (using K-Means and K = 4) -------------------- #
                # if cluster_col == "subdomain_kmeans":
                    
                #     # 1. DE analysis (granule & cell)
                #     for spot_expression, label in zip([spot_granule_expression, spot_cell_expression], ["granule", "cell"]):
                        
                #         spot_expression_adata = anndata.AnnData(X = spot_expression.copy(), obs = spots.obs, var = granule_adata.var)
                #         spot_expression_adata.var_names = granule_adata.var_names
                #         spot_expression_adata.obs_names = spots.obs_names
                        
                #         sc.pp.normalize_total(spot_expression_adata, target_sum = 1e4)
                #         sc.pp.log1p(spot_expression_adata)
                        
                #         sc.tl.rank_genes_groups(spot_expression_adata, cluster_col, method = "t-test")
                #         names = pd.DataFrame(spot_expression_adata.uns["rank_genes_groups"]["names"])
                #         logfc = pd.DataFrame(spot_expression_adata.uns["rank_genes_groups"]["logfoldchanges"])
                #         pvals = pd.DataFrame(spot_expression_adata.uns["rank_genes_groups"]["pvals"])
                #         for cluster in spots.obs[cluster_col].unique():
                #             df = pd.DataFrame({"names": names[cluster], "logfc": logfc[cluster], "pvals": pvals[cluster]})
                #             df = df.sort_values(by = "logfc", ascending = False)    # no filtering here
                #             df.to_csv(output_path + f"{label}_DE_genes_{cluster}.csv", index = False)
                    
                #     # 2. Cluster-specific enrichment analysis
                #     for cluster in spots.obs[cluster_col].unique():
                        
                #         # Get spot-level granule & cell expression
                #         spot_granule_expression_cluster = spot_granule_expression[spots.obs[cluster_col] == cluster].copy()
                #         spot_cell_expression_cluster = spot_cell_expression[spots.obs[cluster_col] == cluster].copy()
                        
                #         # Gene list (both matrices share the same column order)
                #         gene_names = np.asarray(granule_adata.var_names)

                #         for spot_expression_cluster, label in zip([spot_granule_expression_cluster, spot_cell_expression_cluster], ["granule", "cell"]):
                            
                #             if not isinstance(spot_expression_cluster, np.ndarray):
                #                 spot_expression_cluster = spot_expression_cluster.toarray()

                #             # (1) Sort genes by number of spots expressing them
                #             n_spots_cluster = spot_expression_cluster.shape[0]
                #             detection_count = (spot_expression_cluster > 0).sum(axis=0)
                #             detection_frac = detection_count / max(n_spots_cluster, 1)

                #             gene_detection = pd.DataFrame({"gene": gene_names, "detection_count": detection_count, "detection_frac": detection_frac})
                #             gene_detection = gene_detection.sort_values(by=["detection_count", "detection_frac"], ascending=False)
                #             gene_detection.to_csv(output_path + f"{label}_gene_by_detection_{cluster}.csv", index=False)

                #             # (2) Sort genes by expression level (mean / sum across spots)
                #             mean_expr = spot_expression_cluster.mean(axis=0)
                #             sum_expr = spot_expression_cluster.sum(axis=0)
                #             gene_expression = pd.DataFrame({"gene": gene_names, "mean_expression": mean_expr, "sum_expression": sum_expr})
                #             gene_expression = gene_expression.sort_values(by=["mean_expression", "sum_expression"], ascending=False)
                #             gene_expression.to_csv(output_path + f"{label}_gene_by_expression_{cluster}.csv", index=False)